In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
import numpy as np
import matplotlib.pyplot as plt

In [7]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.01):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [8]:
class CNN(nn.Module):
    def __init__(self, config):
        super(CNN, self).__init__()

        self.config = config

        self.conv_layers = nn.Sequential(
            nn.Conv2d(9, 16, kernel_size=3, padding=1), # 9 channels
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.MaxPool2d(2, 2),
        )

        # Makes model patch-size independent
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully connected regression head
        self.fc_layers = nn.Sequential(
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc_layers(x)
        x = torch.sigmoid(x) # Sigmoid activation function

        return x


In [9]:
class CISIDataset(Dataset):
    def __init__(self, gdp_path, pop_path, lc_path, cisi_path):
        # Load all data
        with rasterio.open(gdp_path) as src:
            self.gdp = src.read(1)  # Shape: (H, W)
        
        with rasterio.open(pop_path) as src:
            self.pop = src.read(1)
        
        with rasterio.open(lc_path) as src:
            self.lc = src.read()  # Shape: (7, H, W) - all one-hot channels
        
        with rasterio.open(cisi_path) as src:
            self.cisi = src.read(1)
        
        # Get valid indices (where CISI is not NaN/nodata)
        self.valid_mask = ~np.isnan(self.cisi)
        self.valid_indices = np.argwhere(self.valid_mask)
        
        print(f"Found {len(self.valid_indices)} valid pixels")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        i, j = self.valid_indices[idx]
        
        # 32x32 patch for now
        patch_size = 32
        half_patch = patch_size // 2
        
        # Handle boundaries
        i_start = max(0, i - half_patch)
        i_end = min(self.gdp.shape[0], i + half_patch)
        j_start = max(0, j - half_patch)
        j_end = min(self.gdp.shape[1], j + half_patch)
        
        # Extract patches
        gdp_patch = self.gdp[i_start:i_end, j_start:j_end]
        pop_patch = self.pop[i_start:i_end, j_start:j_end]
        lc_patch = self.lc[:, i_start:i_end, j_start:j_end]
        
        # Pad to patch_size if needed
        pad_i = patch_size - gdp_patch.shape[0]
        pad_j = patch_size - gdp_patch.shape[1]
        
        if pad_i > 0 or pad_j > 0:
            gdp_patch = np.pad(gdp_patch, ((0, pad_i), (0, pad_j)))
            pop_patch = np.pad(pop_patch, ((0, pad_i), (0, pad_j)))
            lc_patch = np.pad(lc_patch, ((0, 0), (0, pad_i), (0, pad_j)))
        
        # Stack: (9, 32, 32)
        input_tensor = np.stack([
            gdp_patch,
            pop_patch,
            *lc_patch  # Unpack all 7 LC channels
        ], axis=0)
        
        # Get target CISI value
        target = self.cisi[i, j]
        
        # Convert to tensors
        input_tensor = torch.FloatTensor(input_tensor)
        target = torch.FloatTensor([target])
        
        return input_tensor, target, (i, j)


In [10]:
def training(
    num_epochs,
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    clip_value,
    device,
    wandbrun=None,
    sigmoidcnn=False
):
    train_loss_list = []
    val_loss_list = []
    early_stopping = EarlyStopping(patience=5, delta=0.01)

    for epoch in range(num_epochs):
        # ========== TRAINING ==========
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            inputs, targets, _ = batch
            inputs = inputs.to(device)
            targets = targets.to(device).view(-1, 1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        train_loss_list.append(avg_train_loss)

        # ========== VALIDATION ==========
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                inputs, targets, _ = batch
                inputs = inputs.to(device)
                targets = targets.to(device).view(-1, 1)

                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_loss_list.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        # Early stopping check
        early_stopping(avg_val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

    return train_loss_list, val_loss_list

# Paths

In [11]:
gdp_path = r"READY_data\inputs\2019_gdp_aligned_025.tif"
pop_path = r"READY_data\inputs\2020_pop_aligned_025.tif"
lc_path = r"READY_data\landuse_onehot_025\clipped_history_2020_025_onehot.tif"
cisi_path = r"READY_data\labels\2024_CISI_025deg.tif"

# Dataset and Split

In [12]:
# Create full dataset
dataset = CISIDataset(gdp_path, pop_path, lc_path, cisi_path)

# Split: 80% train, 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train samples: {train_size}")
print(f"Validation samples: {val_size}")

Found 14922 valid pixels
Train samples: 11937
Validation samples: 2985


# Create Dataloaders

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Init Model

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = {'dropout': 0.3}
model = CNN(config).to(device)

criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cpu


# Train Model

In [15]:
train_loss_list, val_loss_list = training(
    num_epochs=50,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    clip_value=1.0,
    device=device
)

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 16])

# Plotting some results

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Progress')
plt.show()